# Correction RAG — nettoyage des documents Shifaa déjà indexés

**Objectif** : les 34912 chunks Shifaa déjà présents dans la collection ChromaDB (`psych_kb_v1`) ont été indexés avant l'identification du problème de formules d'ouverture patient-facing (basmala, salutation, adresse nominative, bienvenue à la plateforme). Ce notebook les remplace par leur version nettoyée, **sans toucher aux guidelines ni à Kanakmi**.

**Notebook totalement autonome** — ne dépend d'aucune session précédente.


## 1. Installation et connexion

In [ ]:
!pip install -q chromadb sentence-transformers pandas numpy

import os
import re
import gc
import json
import torch
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError(" GPU requis. Activez-le via Runtime > Change runtime type > GPU.")
print(f" GPU : {torch.cuda.get_device_name(0)}")

drive.mount('/content/drive')
BASE_DRIVE = '/content/drive/MyDrive/Master_Thesis_RAG_Indexes'
VECTOR_DB_DIR = f"{BASE_DRIVE}/Vector_Database/chroma_db_v1"
SHIFAA_CSV = f"{BASE_DRIVE}/Shifaa/shifaa_prepared_rag.csv"
print(f" {VECTOR_DB_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:

print(chromadb.__version__)

1.5.9


## 2. Connexion à la collection existante + diagnostic

**Important — étape de sécurité** : avant de supprimer quoi que ce soit, on affiche les valeurs uniques du champ `source` réellement présentes dans la collection, pour confirmer avec certitude laquelle correspond à Shifaa avant de cibler la suppression.

In [ ]:
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_collection("psych_kb_v1")
n_total_before = collection.count()
print(f"✅ Collection connectée : {n_total_before} documents au total")

# Échantillonne les métadonnées pour identifier les valeurs uniques de 'source'
sample = collection.get(limit=min(2000, n_total_before), include=["metadatas"])
unique_sources = set(m.get("source", "?") for m in sample["metadatas"])
print("\nValeurs uniques de 'source' observées dans cet échantillon :")
for s in sorted(unique_sources):
    print(f"  - {s}")
print("\n⚠️ Identifiez ci-dessus la valeur exacte correspondant à Shifaa avant de continuer,")
print("   et ajustez SHIFAA_SOURCE_VALUE dans la cellule suivante si besoin.")

✅ Collection connectée : 51854 documents au total

Valeurs uniques de 'source' observées dans cet échantillon :
  - APA_PTSD_2025_EN
  - APA_Schizophrenia_2020_EN
  - CANMAT_ISBD_Bipolar_2025_EN
  - Kanakmi/mental-disorders
  - NICE_CG185_Bipolar_2024_EN
  - NICE_CG78_BPD_2009_EN
  - NICE_NG222_Depression_2022_EN
  - NICE_NG222_Depression_2022_FR
  - VA_DoD_PTSD_2023_EN
  - mhGAP_AR
  - mhGAP_EN
  - mhGAP_FR

⚠️ Identifiez ci-dessus la valeur exacte correspondant à Shifaa avant de continuer,
   et ajustez SHIFAA_SOURCE_VALUE dans la cellule suivante si besoin.


In [ ]:
SHIFAA_SOURCE_VALUE = "Shifaa_Arabic_Mental_Health_Consultations"  # confirmé via content_type="clinical_dialogue"

existing_shifaa = collection.get(where={"source": SHIFAA_SOURCE_VALUE}, include=[])
n_shifaa_existing = len(existing_shifaa["ids"])
print(f"✅ {n_shifaa_existing} documents Shifaa trouvés dans la collection avec source='{SHIFAA_SOURCE_VALUE}'")
if n_shifaa_existing == 0:
    print("❌ Aucun document trouvé avec cette valeur — corrigez SHIFAA_SOURCE_VALUE d'après le diagnostic ci-dessus et relancez cette cellule.")

✅ 34912 documents Shifaa trouvés dans la collection avec source='Shifaa_Arabic_Mental_Health_Consultations'


## 3. Reconstruction des documents Shifaa nettoyés

On ré-extrait Diagnosis/Patient/Doctor depuis le CSV source, on nettoie uniquement la partie "réponse", et on reconstruit le document Markdown final dans le même format que l'original — seul le contenu de `## Doctor` change.

In [ ]:
df = pd.read_csv(SHIFAA_CSV)
print(f"✅ {len(df)} lignes chargées depuis {SHIFAA_CSV}")

def extract_qa_pair(text):
    q_match = re.search(r"## Patient\s*\n(.*?)\n\n## Doctor", text, re.DOTALL)
    a_match = re.search(r"## Doctor\s*\n(.*)", text, re.DOTALL)
    if not (q_match and a_match):
        return None, None
    return q_match.group(1).strip(), a_match.group(1).strip()

def clean_answer_boilerplate(text):
    """Version finale validée — identique à celle utilisée pour la préparation des
    données de fine-tuning LoRA, pour garantir la cohérence entre RAG et LoRA."""
    text = text.strip()
    patterns = [
        r"^بسم الله الرحمن الرحيم\.?\s*",
        r"^(الأخ|الاخ|أخي|الأخت|الاخت)\s+الفاضل[ةه]?\s*/?\s*[^.\n]{0,60}?(حفظه الله|حفظها الله)\.?\s*",
        r"^(ابننا|ابنتنا|بنتنا|اخونا|اختنا)\s+الفاضل[ةه]?[،,]\s*",
        r"^(أيها|ايها|أيتها|ايتها)\s+(الفاضل[ةه]?|الكريم[ةه]?)(\s+(الفاضل[ةه]?|الكريم[ةه]?))?\s*:\s*",
        r"^(السلام عليكم|وعليكم السلام)(\s+ورحمة الله(\s+تعالى)?)?(\s+وبركاته)?\.?\s*",
        r"^بارك الله فيك[،,]?\s*وجزاك الله خيرا[،,]?\s*",
        r"^جزاك الله خيرا على سؤالك[،,]?\.?\s*",
        r"^وبعد\s*[:,،]*\s*",
        r"^(ف)?(نرحب بك(م)?|مرحبا بك(م)?)\s*([-–—][^-–—]{0,50}[-–—])?\s*(في|عبر|إلى)\s+[^.]*\.\s*",
        r"^شكرا لك على التواصل معنا[،,]*\s*فكرتك وصلت بشكل جيد\.?\s*",
        r"^[.,،\-–—]\s*",
    ]
    changed = True
    while changed:
        changed = False
        for p in patterns:
            new_text = re.sub(p, "", text)
            if new_text != text:
                text = new_text.strip()
                changed = True

    GREETING_KEYWORDS = [
        "نرحب", "مرحبا", "السلام عليكم", "وعليكم السلام", "بسم الله",
        "الفاضل", "الفاضلة", "حفظه الله", "حفظها الله", "بارك الله",
        "جزاك الله", "نشكر لك", "شكرا لك", "تواصلك", "التواصل معنا",
        "ابننا", "ابنتنا", "بنتنا", "اختنا", "أختنا",
    ]
    for _ in range(3):
        first_period = text.find(".")
        if first_period == -1 or first_period > 250:
            break
        first_sentence = text[:first_period + 1]
        if any(kw in first_sentence for kw in GREETING_KEYWORDS):
            text = text[first_period + 1:].strip()
        else:
            break
    return text

qa_pairs = df["text"].apply(extract_qa_pair)
df["question_extracted"] = [p[0] for p in qa_pairs]
df["answer_raw"] = [p[1] for p in qa_pairs]
df["answer_clean"] = df["answer_raw"].apply(lambda x: clean_answer_boilerplate(x) if isinstance(x, str) else x)

n_valid = df["answer_clean"].notna().sum()
n_changed = (df["answer_clean"] != df["answer_raw"]).sum()
print(f"✅ {n_valid}/{len(df)} lignes extraites avec succès")
print(f"🧹 {n_changed} réponses nettoyées d'une formule d'ouverture")

✅ 34912 lignes chargées depuis /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Shifaa/shifaa_prepared_rag.csv
✅ 34912/34912 lignes extraites avec succès
🧹 34905 réponses nettoyées d'une formule d'ouverture


In [ ]:
def rebuild_document(row):
    """Reconstruit le document Markdown final dans le MÊME format que l'original
    (## Diagnosis / ## Patient / ## Doctor) — seule la partie Doctor change."""
    if pd.isna(row["answer_clean"]) or pd.isna(row["question_extracted"]):
        return row["text"] 
    return (
        f"**Language:** Arabic\n\n"
        f"## Diagnosis\n{row['disorder']}\n\n"
        f"## Patient\n{row['question_extracted']}\n\n"
        f"## Doctor\n{row['answer_clean']}"
    )

df["text_clean"] = df.apply(rebuild_document, axis=1)
print("✅ Documents reconstruits")
print("\n--- Exemple de document final ---")
print(df["text_clean"].iloc[0][:400])

✅ Documents reconstruits

--- Exemple de document final ---
**Language:** Arabic

## Diagnosis
الحالات النفسية السلوكية

## Patient
السلام عليكم ورحمة الله وبركاتهاعاني منذ سنوات من تخيل بعض المواقف في ذهني، حول خلافي او شجاري مع شخص ما، واشعر بانفعال شديد بسبب هذه الخيالات، لدرجة تصل الى ارتفاع معدل ضربات القلب بشكل ملحوظ، وارتفاع الضغط، فذهبت الى طبيب القلب، ووصف لي دواء الاندرال، ولكني ما زلت اعاني من الخيالات، فما سببها؟ وهل لها علاج؟

## Doctor
ظهور خ


## 4. Remplacement dans ChromaDB — suppression puis ré-ajout

Les anciens documents Shifaa sont supprimés par leur identifiant exact (les mêmes IDs que ceux déjà présents dans la collection, issus de la colonne `id` du CSV), puis ré-ajoutés avec le texte nettoyé et de nouveaux embeddings — les guidelines et Kanakmi ne sont jamais touchés.

In [ ]:
ids_to_replace = df["id"].astype(str).tolist()

# Vérifie que ces IDs existent bien dans la collection avant de supprimer quoi que ce soit
check = collection.get(ids=ids_to_replace[:5], include=[])
print(f"Vérification sur un échantillon de 5 IDs : {len(check['ids'])}/5 trouvés dans la collection")
if len(check["ids"]) == 0:
    raise RuntimeError("❌ Aucun des IDs attendus n'a été trouvé — vérifiez le format de la colonne 'id' du CSV avant de continuer.")

Vérification sur un échantillon de 5 IDs : 5/5 trouvés dans la collection


In [ ]:
print("⏳ Suppression des anciens documents Shifaa...")
batch_size_delete = 500
for i in range(0, len(ids_to_replace), batch_size_delete):
    batch_ids = ids_to_replace[i:i+batch_size_delete]
    collection.delete(ids=batch_ids)
print(f"✅ {len(ids_to_replace)} anciens documents supprimés")
print(f"   Documents restants dans la collection : {collection.count()} (guidelines + Kanakmi, Shifaa temporairement absent)")

⏳ Suppression des anciens documents Shifaa...
✅ 34912 anciens documents supprimés
   Documents restants dans la collection : 16942 (guidelines + Kanakmi, Shifaa temporairement absent)


In [ ]:
print("⏳ Chargement du modèle d'embedding (bge-m3)...")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda").half()
print("✅ Modèle chargé")

print("⏳ Génération des nouveaux embeddings...")
texts = df["text_clean"].tolist()
embeddings = embed_model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print(f"✅ {len(embeddings)} embeddings générés")

⏳ Chargement du modèle d'embedding (bge-m3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Modèle chargé
⏳ Génération des nouveaux embeddings...


Batches:   0%|          | 0/546 [00:00<?, ?it/s]

✅ 34912 embeddings générés


In [ ]:
print(" Ré-ajout des documents nettoyés dans ChromaDB...")
batch_size_add = 500
for i in range(0, len(df), batch_size_add):
    batch = df.iloc[i:i+batch_size_add]
    batch_embeddings = embeddings[i:i+batch_size_add]
    collection.add(
        ids=batch["id"].astype(str).tolist(),
        embeddings=[e.tolist() for e in batch_embeddings],
        documents=batch["text_clean"].tolist(),
        metadatas=[
            {
                "lang": row["lang"],
                "role_clinical": bool(row["role_clinical"]),
                "role_education": bool(row["role_education"]),
                "content_type": row["content_type"],
                "source": row["source"],
                "disorder": row["disorder"],
            }
            for _, row in batch.iterrows()
        ],
    )
    print(f"  [{min(i+batch_size_add, len(df))}/{len(df)}] documents ré-ajoutés")

print(f"\n Ré-ajout terminé")

 Ré-ajout des documents nettoyés dans ChromaDB...
  [500/34912] documents ré-ajoutés
  [1000/34912] documents ré-ajoutés
  [1500/34912] documents ré-ajoutés
  [2000/34912] documents ré-ajoutés
  [2500/34912] documents ré-ajoutés
  [3000/34912] documents ré-ajoutés
  [3500/34912] documents ré-ajoutés
  [4000/34912] documents ré-ajoutés
  [4500/34912] documents ré-ajoutés
  [5000/34912] documents ré-ajoutés
  [5500/34912] documents ré-ajoutés
  [6000/34912] documents ré-ajoutés
  [6500/34912] documents ré-ajoutés
  [7000/34912] documents ré-ajoutés
  [7500/34912] documents ré-ajoutés
  [8000/34912] documents ré-ajoutés
  [8500/34912] documents ré-ajoutés
  [9000/34912] documents ré-ajoutés
  [9500/34912] documents ré-ajoutés
  [10000/34912] documents ré-ajoutés
  [10500/34912] documents ré-ajoutés
  [11000/34912] documents ré-ajoutés
  [11500/34912] documents ré-ajoutés
  [12000/34912] documents ré-ajoutés
  [12500/34912] documents ré-ajoutés
  [13000/34912] documents ré-ajoutés
  [13500